# Predicting Band Gaps of SnSe-Based Materials from Composition

**Capstone project — Machine Learning for Materials and Metallurgical Engineering (UGC Malaviya Mission Teacher Training Programme)**

**Why SnSe:** single-crystal SnSe is one of the best-performing thermoelectric materials discovered in the last decade, and its electronic structure — especially how the band gap shifts under doping and alloying (Pb, Ge, S, Te, Ag, Na, halides, ...) — is an actively studied "band engineering" question. A composition-only ML model is a fast way to screen which dopants/alloys are predicted to open or close the gap, before running expensive DFT.

**Objective:** Query the Materials Project for Sn-Se-based compounds (compounds containing both Sn and Se, plus any dopant/alloying elements), featurize them from composition alone (matminer/Magpie), and:
1. **Classify** metal vs. insulator (`band_gap == 0` vs. `> 0`) — Logistic Regression vs. Random Forest.
2. **Regress** the band gap value for insulators — Random Forest regression, with feature-importance interpretation.

**Verified dataset size (checked live against the API, not an estimate):** 220 raw entries → 192 after de-duplication/cleaning → 132 Magpie features → 136 insulators / 56 metals. This is a genuinely small dataset relative to the feature count, so the notebook uses 5-fold cross-validation (not just a single train/test split) for the headline metrics — a single split at N≈190 is noisy. The small-N/high-dimensionality tension is itself worth a sentence in the report.

**"Done" looks like:** a cross-validated classifier and regressor, a feature-importance plot you can discuss in materials terms (e.g. electronegativity difference), and 2-3 figures ready to drop into the report/slides.

**Team roles for this notebook:**
- Data & Preprocessing Lead: Steps 1-2
- Modeling Lead: Steps 3-6
- Everyone: Step 7 (interpretation feeds the report/slides)

**Before running:** get a free API key at https://next-gen.materialsproject.org/api (32-character key). Do **not** hardcode it in this notebook — set it as an environment variable instead (see Setup).

## Setup

Install once (uncomment if needed), then set your API key as an environment variable in your terminal **before** launching Jupyter:

```bash
export MP_API_KEY="your_32_char_key_here"
jupyter lab
```

In [ ]:
# %pip install mp-api pymatgen matminer scikit-learn pandas matplotlib

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, r2_score

RANDOM_STATE = 42

## Step 1 — Query the Materials Project

Pull all compounds that contain **both Sn and Se** (binary SnSe/SnSe2/Sn2Se3, plus any ternary/quaternary/quinary doped or alloyed derivatives — e.g. Sn-Se-Pb, Sn-Se-S, Sn-Se-Ag). This uses the current `mp-api` client — not the deprecated `pymatgen.ext.matproj.MPRester().query()` syntax seen in older tutorials.

**Verified live against the API:** the breakdown by element count is 11 binary (Sn-Se only) + 83 ternary + 92 quaternary + 30 quinary + 4 six-element = **220 total**, essentially all of it within 2-5 elements (`num_elements=(2, 5)` below captures 216/220; going to 7 catches the rest). This is smaller than a typical ML dataset — see the note at the top of the notebook on why cross-validation is used instead of a single train/test split.

In [ ]:
from mp_api.client import MPRester

api_key = os.environ["MP_API_KEY"]  # raises a clear error if you forgot to set it

with MPRester(api_key=api_key) as mpr:
    docs = mpr.materials.summary.search(
        elements=["Sn", "Se"],   # must contain BOTH — other elements (dopants/alloys) allowed
        num_elements=(2, 7),      # covers the full verified set (up to 6-element compounds)
        fields=["material_id", "formula_pretty", "band_gap"],
    )

raw = pd.DataFrame([d.model_dump() for d in docs])
print(f"{len(raw)} entries returned")  # expect 220
raw.head()

## Step 2 — Clean and build compositions

Drop any duplicate formulas or missing band gaps, then turn each formula into a `pymatgen.core.Composition` object — matminer's featurizers need this, not a raw string.

In [ ]:
df = raw.dropna(subset=["formula_pretty", "band_gap"]).drop_duplicates(subset="formula_pretty").copy()
df["composition"] = df["formula_pretty"].apply(Composition)

print(f"{len(df)} unique compounds after cleaning")
df[["formula_pretty", "band_gap"]].describe()

## Step 3 — Featurize with matminer (Magpie)

`ElementProperty.from_preset("magpie")` turns each composition into ~130 descriptors (mean/range/etc. of electronegativity, atomic radius, valence electron count, ...). This is composition-only — no crystal structure needed — which keeps the project tractable in a week.

In [ ]:
featurizer = ElementProperty.from_preset("magpie")
df = featurizer.featurize_dataframe(df, col_id="composition", ignore_errors=True)

feature_cols = featurizer.feature_labels()
df = df.dropna(subset=feature_cols)
print(f"{len(df)} compounds x {len(feature_cols)} features after featurization")

## Step 4 — Define the classification target

`is_metal = 1` when `band_gap == 0`, else `0`. Split into train/test once here and reuse the same split for both models so results are comparable.

In [ ]:
df["is_metal"] = (df["band_gap"] == 0).astype(int)
print(df["is_metal"].value_counts(normalize=True))

X = df[feature_cols]
y_class = df["is_metal"]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y_class, df.index, test_size=0.2, random_state=RANDOM_STATE, stratify=y_class
)

## Step 5 — Classification: Logistic Regression vs. Random Forest

Compare a simple linear baseline against Random Forest. Logistic Regression needs scaled features; Random Forest does not.

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
logreg.fit(X_train_scaled, y_train)
logreg_acc = accuracy_score(y_test, logreg.predict(X_test_scaled))

rf_clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
rf_clf.fit(X_train, y_train)
rf_clf_acc = accuracy_score(y_test, rf_clf.predict(X_test))

print(f"Logistic Regression accuracy: {logreg_acc:.3f}")
print(f"Random Forest accuracy:       {rf_clf_acc:.3f}")
print("\nRandom Forest confusion matrix (rows=true, cols=predicted):")
print(confusion_matrix(y_test, rf_clf.predict(X_test)))

### Step 5b — cross-validated accuracy (the number to actually report)

With only ~190 compounds, a single 80/20 split's accuracy can swing several points depending on which rows land in the test set. 5-fold cross-validation averages over 5 different splits — report this mean ± std in the report/slides, not the single-split number above.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_scaled_full = StandardScaler().fit_transform(X)

logreg_cv = cross_val_score(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), X_scaled_full, y_class, cv=skf)
rf_cv = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE), X, y_class, cv=skf)

print(f"Logistic Regression 5-fold accuracy: {logreg_cv.mean():.3f} +/- {logreg_cv.std():.3f}")
print(f"Random Forest       5-fold accuracy: {rf_cv.mean():.3f} +/- {rf_cv.std():.3f}")

## Step 6 — Regression: Random Forest on insulators

Restrict to insulators (`band_gap > 0`) and predict the actual gap value. Report MAE (in eV, easy to interpret) and R².

In [ ]:
insulators = df[df["band_gap"] > 0]
X_ins = insulators[feature_cols]
y_ins = insulators["band_gap"]

Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    X_ins, y_ins, test_size=0.2, random_state=RANDOM_STATE
)

rf_reg = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)
rf_reg.fit(Xi_train, yi_train)
y_pred = rf_reg.predict(Xi_test)

print(f"MAE: {mean_absolute_error(yi_test, y_pred):.3f} eV")
print(f"R^2: {r2_score(yi_test, y_pred):.3f}")

plt.figure(figsize=(5, 5))
plt.scatter(yi_test, y_pred, alpha=0.4, s=15)
lims = [0, max(yi_test.max(), y_pred.max())]
plt.plot(lims, lims, "k--", linewidth=1)
plt.xlabel("True band gap (eV)")
plt.ylabel("Predicted band gap (eV)")
plt.title("Random Forest: predicted vs. true band gap")
plt.tight_layout()
plt.show()

### Step 6b — cross-validated regression metrics (the numbers to actually report)

Same reasoning as 5b: with 136 insulator rows, a single test split of ~27 compounds gives a noisy MAE/R². Report the 5-fold mean ± std instead.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_reg_cv = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)

mae_cv = -cross_val_score(rf_reg_cv, X_ins, y_ins, cv=kf, scoring="neg_mean_absolute_error")
r2_cv = cross_val_score(rf_reg_cv, X_ins, y_ins, cv=kf, scoring="r2")

print(f"Random Forest 5-fold MAE: {mae_cv.mean():.3f} +/- {mae_cv.std():.3f} eV")
print(f"Random Forest 5-fold R^2: {r2_cv.mean():.3f} +/- {r2_cv.std():.3f}")

## Step 7 — Feature importance (interpretation)

Which composition descriptors drive the model's predictions? Discuss the top features in materials terms in the report — e.g. if electronegativity-related features dominate, that ties back to ionic/covalent bonding character.

In [ ]:
importances = pd.Series(rf_reg.feature_importances_, index=feature_cols).sort_values(ascending=False)
top_n = importances.head(15)

plt.figure(figsize=(7, 6))
top_n.iloc[::-1].plot(kind="barh")
plt.xlabel("Feature importance")
plt.title("Top 15 features — band gap regression")
plt.tight_layout()
plt.show()

top_n

## Step 8 (optional, if time allows) — PCA visualization

Compress the ~130 Magpie features to 2D and color by band gap, to see whether the Sn-Se compound chemistry (base binary vs. various dopants/alloys) separates into visually sensible regions. Reuses the Day 3 PCA/clustering material — treat as a stretch goal, not required.

In [ ]:
from sklearn.decomposition import PCA

X_scaled_full = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled_full)

plt.figure(figsize=(6, 5))
sc = plt.scatter(coords[:, 0], coords[:, 1], c=df["band_gap"], cmap="viridis", s=12, alpha=0.7)
plt.colorbar(sc, label="Band gap (eV)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
plt.title("PCA of Magpie composition features, colored by band gap")
plt.tight_layout()
plt.show()

## Conclusions (fill in for the report)

- Classification accuracy (Logistic Regression vs. Random Forest): ...
- Regression MAE / R^2, and how that compares to SnSe's known ~0.86 eV gap and the spread seen across doped/alloyed derivatives: ...
- Which features mattered most, and does that make physical sense for a IV-VI narrow-gap semiconductor (e.g. electronegativity/atomic-radius mismatch from the dopant driving gap shifts): ...
- Limitations: composition-only features ignore crystal structure/polymorphism, so two compounds with the same formula but different structures (e.g. SnSe's orthorhombic *Pnma* vs. its high-temperature phase) get identical features — mention this as a known limitation in the report. Also note the dataset mixes very different "families" (pure SnSe polymorphs vs. heavily doped ternaries/quaternaries), which the report should call out.
- Next steps if more time were available: structure-based features (e.g. via `matminer`'s structure featurizers) to distinguish polymorphs, hyperparameter tuning, or extending to the full IV-VI chalcogenide family (SnS, SnTe, PbSe, GeSe, ...) for comparison.